In [1]:
import pandas as pd

df = pd.read_csv("../data/papers_clustered.csv")
df["abstract"] = df["abstract"].fillna("")

MEASURE = ["kit test", "kit value", "kit rating", "cobb", "wvtr",
           "water vapor transmission", "water vapour transmission",
           "contact angle", "grease resistance", "oil resistance",
           "coat weight", "coating weight", "g/m"]

def measure_score(row):
    text = f"{row['title']} {row['abstract']}".lower()
    return sum(k in text for k in MEASURE)

df["measure_score"] = df.apply(measure_score, axis=1)

cand = df[(df["is_oa"] == True) & (df["type"] == "article")]

shortlist = (cand.sort_values(["measure_score", "cited_by"], ascending=False)
                 .groupby("cluster_name").head(6))

for name, sub in shortlist.groupby("cluster_name"):
    print(f"\n=== {name} ===")
    for _, r in sub.iterrows():
        print(f"  [score {r['measure_score']}] {r['year']} | {r['title'][:90]}")

shortlist.to_csv("../data/shortlist.csv", index=False, encoding="utf-8")
print("\nSaved: data/shortlist.csv")


=== Chitosan coatings ===
  [score 5] 2023 | Barrier and Antimicrobial Properties of Coatings Based on Xylan Derivatives and Chitosan f
  [score 5] 2024 | Impact of Silver Nanoparticle Treatment and Chitosan on Packaging Paper’s Barrier Effectiv
  [score 4] 2012 | Application of Chitosan as a Barrier Coating on Coated Ivory Board
  [score 4] 2025 | Impact of Surface Tension and Surface Energy on Spray Coating Paper with Polysaccharide-Ba
  [score 4] 2026 | Biopolymer Multilayer Structure as a Barrier System for Packaging Paper
  [score 3] 2018 | Application of Industrially Produced Chitosan in the Surface Treatment of Fibre-Based Mate

=== Edible & antimicrobial films ===
  [score 2] 2026 | Glass-spread fabrication of ultralow-loading ultrathin PLA coatings for improving Water re
  [score 1] 2025 | High-performance edible alginate straws constructed with zein coatings by phase separation
  [score 0] 2015 | State of the Art in the Development and Properties of Protein-Based Films and C

In [2]:
SELECTED = [
    "Application of Chitosan as a Barrier Coating on Coated Ivory Board",
    "Coatings Based on Xylan Derivatives and Chitosan",
    "Microfibrillated Cellulose Based Barrier Coatings for Abrasive Paper",
    "Functional barrier and recyclable packaging materials through microfibrillated",
    "Addition of bentonite to cationic starch matrix",
    "Starch - Sodium Alginate Composite Coating",
    "Wax Coatings for Paper Packaging Applications",
    "Degradable Polymeric Waxes for Paper Coating",
]

mask = shortlist["title"].apply(
    lambda t: any(s.lower() in t.lower() for s in SELECTED)
)
selected = shortlist[mask]

print("Selected:", len(selected), "/", len(SELECTED))
for i, (_, r) in enumerate(selected.iterrows(), 1):
    print(f"\n{i}. {r['title']}")
    print(f"   {r['oa_url']}")

selected.to_csv("../data/selected_papers.csv", index=False, encoding="utf-8")

Selected: 8 / 8

1. Barrier and Antimicrobial Properties of Coatings Based on Xylan Derivatives and Chitosan for Food Packaging Papers
   https://doi.org/10.3390/coatings13101761

2. Microfibrillated Cellulose Based Barrier Coatings for Abrasive Paper Products
   https://www.mdpi.com/2079-6412/10/11/1108/pdf?version=1605773187

3. Addition of bentonite to cationic starch matrix for coating on kraftliner paper to improve grease resistance
   https://curis.ku.dk/ws/files/422280587/Addition_of_bentonite_to_cationic_starch_matrix_for_coating_on_kraftliner_paper_to_improve_grease_resistance.pdf

4. The Application of Starch - Sodium Alginate Composite Coating on Transparent Paper for Food Packaging
   https://doi.org/10.4028/www.scientific.net/amr.893.472

5. Application of Chitosan as a Barrier Coating on Coated Ivory Board
   https://doi.org/10.4028/www.scientific.net/amm.200.180

6. Degradable Polymeric Waxes for Paper Coating Applications
   https://doi.org/10.1021/acsapm.3c03072

7. Fu